In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

CWD = Path.cwd()

if (CWD / "data").exists():
    BASE_DIR = CWD
else:
    BASE_DIR = CWD.parent

DATA_RAW = BASE_DIR / "data" / "raw"
DATABASE_DIR = BASE_DIR / "data" / "database"

DATABASE_DIR.mkdir(parents=True, exist_ok=True)

CSV_PACIENTES = DATA_RAW / "pacientes.csv"
DB_PATH = DATABASE_DIR / "hospital.db"

print("Base do projeto:", BASE_DIR)
print("Banco:", DB_PATH)

Base do projeto: C:\Users\Diogo\tech-challenge-FIAP-fase3-assistente-medico
Banco: C:\Users\Diogo\tech-challenge-FIAP-fase3-assistente-medico\data\database\hospital.db


In [2]:
# ============================================================
# CÉLULA 2 — Carregar ou recriar base sintética de pacientes
# ============================================================

DATA_RAW.mkdir(parents=True, exist_ok=True)

if not CSV_PACIENTES.exists():

    print("pacientes.csv não encontrado.")
    print("Recriando base sintética...")

    pacientes = [
        {
            "id_paciente": "PAC001",
            "idade": 52,
            "sexo": "F",
            "historico_familiar": "Sim",
            "resultado_exame": "Achado suspeito",
            "exames_pendentes": "Ultrassonografia complementar",
            "status": "Em investigação"
        },
        {
            "id_paciente": "PAC002",
            "idade": 44,
            "sexo": "F",
            "historico_familiar": "Não",
            "resultado_exame": "Sem alterações suspeitas",
            "exames_pendentes": "Nenhum",
            "status": "Acompanhamento"
        },
        {
            "id_paciente": "PAC003",
            "idade": 61,
            "sexo": "F",
            "historico_familiar": "Sim",
            "resultado_exame": "Achado inconclusivo",
            "exames_pendentes": "Avaliação especializada",
            "status": "Em investigação"
        },
        {
            "id_paciente": "PAC004",
            "idade": 39,
            "sexo": "F",
            "historico_familiar": "Não",
            "resultado_exame": "Achado benigno",
            "exames_pendentes": "Nenhum",
            "status": "Acompanhamento"
        }
    ]

    df_pacientes = pd.DataFrame(pacientes)

    df_pacientes.to_csv(
        CSV_PACIENTES,
        index=False,
        encoding="utf-8-sig"
    )

    print("Arquivo recriado com sucesso:")
    print(CSV_PACIENTES)

else:

    print("Arquivo encontrado:")
    print(CSV_PACIENTES)

df_pacientes = pd.read_csv(CSV_PACIENTES)

display(df_pacientes)

Arquivo encontrado:
C:\Users\Diogo\tech-challenge-FIAP-fase3-assistente-medico\data\raw\pacientes.csv


,id_paciente,idade,sexo,historico_familiar,resultado_exame,exames_pendentes,status
0,PAC001,52,F,Sim,Achado suspeito,Ultrassonografia complementar,Em investigação
1,PAC002,44,F,Não,Sem alterações suspeitas,Nenhum,Acompanhamento
2,PAC003,61,F,Sim,Achado inconclusivo,Avaliação especializada,Em investigação
3,PAC004,39,F,Não,Achado benigno,Nenhum,Acompanhamento


In [3]:
conn = sqlite3.connect(DB_PATH)

df_pacientes.to_sql(
    "pacientes",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

print("Banco criado com sucesso.")

Banco criado com sucesso.


In [7]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri(
    f"sqlite:///{DB_PATH.as_posix()}"
)

print("Tabelas disponíveis:")
print(db.get_usable_table_names())

Tabelas disponíveis:
['pacientes']


In [5]:
%pip install -U langchain langchain-community langchain-huggingface langgraph sqlalchemy

  Using cached langchain-1.4.0-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
  Using cached langgraph-1.2.11-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_core-1.6.3-py3-none-any.whl.metadata (4.8 kB)
  Using cached langgraph_checkpoint-4.2.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.4.4-py3-none-any.whl.metadata (5.1 kB)
  Using cached langchain_protocol-0.0.19-py3-none-any.whl.metadata (2.4 kB)
  Using cached langsmith-0.12.4-py3-none-any.whl.metadata (22 kB)
  Using cached httpx2-2.12.0-py3-none-any.whl.metadata (9.5 kB)
  Using cached httpcore2-2.12.0-py3-none-any.whl.metadata (25 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached langchain_classic-1.0.8-py3-none-any.whl.metadata (5.1 kB)
  Using 

In [6]:
import sys
import langchain_community

print("Python do notebook:")
print(sys.executable)

print("\nLangChain Community:")
print(langchain_community.__version__)

Python do notebook:
C:\Users\Diogo\anaconda3\envs\fiap_fase3\python.exe

LangChain Community:
0.4.2


C:\Users\Diogo\AppData\Local\Temp\ipykernel_11580\1518411597.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


In [8]:
consulta = """
SELECT
    id_paciente,
    idade,
    historico_familiar,
    resultado_exame,
    exames_pendentes,
    status
FROM pacientes
WHERE id_paciente = 'PAC001';
"""

resultado = db.run(consulta)

print(resultado)

[('PAC001', 52, 'Sim', 'Achado suspeito', 'Ultrassonografia complementar', 'Em investigação')]


In [10]:
from langchain_core.tools import tool

@tool
def consultar_paciente(id_paciente: str) -> str:
    """
    Consulta os dados clínicos de um paciente pelo ID.
    """

    query = f"""
    SELECT
        id_paciente,
        idade,
        sexo,
        historico_familiar,
        resultado_exame,
        exames_pendentes,
        status
    FROM pacientes
    WHERE id_paciente = '{id_paciente}'
    """

    resultado = db.run(query)

    if not resultado or resultado == "":
        return "Paciente não encontrado."

    return resultado

In [11]:
resultado = consultar_paciente.invoke("PAC001")

print(resultado)

[('PAC001', 52, 'F', 'Sim', 'Achado suspeito', 'Ultrassonografia complementar', 'Em investigação')]
